In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Ichthyotoxic dataset integration and label-consistency analysis
- This notebook integrates peptide-level ichthyotoxicity annotations derived from AMPDB v1 into a unified, curated dataset. The input consists of a preprocessed CSV file containing peptide sequences and binary ichthyotoxic labels.

- All unique peptide sequences are collected and used to construct a pivot table in which each row represents a unique sequence and each column corresponds to a data source. Since only one source is available, the pivot structure is kept consistent with other toxicity tasks to ensure methodological uniformity.

- Sequence-level quality control is applied by removing peptides containing non-canonical amino acids and by filtering sequences outside predefined minimum and maximum length thresholds. Statistics describing the impact of these filters and the resulting length distribution are recorded.

- Source-specific labels are mapped onto the pivot table using a standardized encoding scheme that distinguishes positive, negative, unlabeled, and unknown annotations. Label consistency is evaluated by counting per-sequence label occurrences and computing the proportion of positive and negative evidence.

- Based on this analysis, sequences are categorized into exclusive positive, exclusive negative, unlabeled-only, or ambiguous groups. Ambiguous cases are further stratified according to the percentage of positive annotations to provide a graded confidence assessment.

- Finally, curated output subsets and comprehensive metadata are generated and exported in a structured format, enabling reproducible downstream analysis and direct use in machine learning workflows.

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/ichthyotoxic"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_AMPDB_ichthyotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/AMPDB v1/processed_ichthyotoxic_dataset.csv")
df_AMPDB_ichthyotoxic = df_AMPDB_ichthyotoxic.rename(columns={"label": "ichthyotoxic"})

- Collecting all sequences for activity

In [4]:
df_list_ichthyotoxic = [
    df_AMPDB_ichthyotoxic
]
unique_sequence_ichthyotoxic = count_unique_sequence(df_list_ichthyotoxic)

5


- Create pivote dataset

In [5]:
df_pivote = create_pivote(unique_sequence_ichthyotoxic)

- Removing sequences with non canonical residues 

In [6]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [7]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True    5
Name: count, dtype: int64


- Filter sequences by length

In [8]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count     5.000000
mean     22.200000
std       5.167204
min      13.000000
25%      24.000000
50%      24.000000
75%      25.000000
max      25.000000
Name: length, dtype: float64

In [9]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [10]:
df_pivote["filter_length"].value_counts()

filter_length
True    5
Name: count, dtype: int64

In [11]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [12]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(5, 4)

In [13]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [14]:
df_list_ichthyotoxic = [("AMPDB", df_AMPDB_ichthyotoxic)]

In [15]:
for source, dataset in df_list_ichthyotoxic:
    dataset = dataset[["sequence", "ichthyotoxic"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["ichthyotoxic"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [16]:
df_pivote

,sequence,AMPDB
0,FIGGIISFFKRLF,1
1,LFGFLIPLLPHLIGAIPQVIGAIR,1
2,LFGFLIKLIPSLFGALSNIGRNRNQ,1
3,LFGFLIPLLPHIIGAIPQVIGAIR,1
4,NWRKILGQIASVGAGLLGSLLAGYE,1


- Working with pivote for detecting ambiguous sequences 

In [17]:
df_pivote = process_count_labels(df_pivote)

In [18]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
False    5
Name: count, dtype: int64

In [19]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
False    5
Name: count, dtype: int64

In [20]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
True    5
Name: count, dtype: int64

In [21]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
True    5
Name: count, dtype: int64

In [22]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    5
Name: count, dtype: int64

In [23]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,AMPDB,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
0,FIGGIISFFKRLF,1,1,0,0,0,True,False,True,False,False,0.0,100.0
1,LFGFLIPLLPHLIGAIPQVIGAIR,1,1,0,0,0,True,False,True,False,False,0.0,100.0
2,LFGFLIKLIPSLFGALSNIGRNRNQ,1,1,0,0,0,True,False,True,False,False,0.0,100.0
3,LFGFLIPLLPHIIGAIPQVIGAIR,1,1,0,0,0,True,False,True,False,False,0.0,100.0
4,NWRKILGQIASVGAGLLGSLLAGYE,1,1,0,0,0,True,False,True,False,False,0.0,100.0


- Splitting data into only negative, only positive, and with amiguous data

In [24]:
negative = df_pivote[df_pivote["negative"]]

In [25]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [26]:
positive = df_pivote[df_pivote["positive"]]

In [27]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [28]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [29]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [30]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [31]:
df_ambiguous["Category_pbb"].value_counts()

Series([], Name: count, dtype: int64)

- Working with metada

In [32]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="ichthyotoxic",
    source_list=df_list_ichthyotoxic,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)
metadata

{'task': 'ichthyotoxic',
 'generated_at': '2026-07-24T16:23:52.142837',
 'sources': {'n_unique_sequences': {'AMPDB': 5}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 5, 'after': 5},
  'length_filter': {'before': 5, 'after': 5},
  'length_distribution': {'min': 13, 'max': 25, 'mean': 22.2, 'median': 24.0}},
 'statistics': {'total_sequences_final': 5,
  'positive': {'positive_and_unlabel': 5, 'only_positive': 5},
  'negative': {'negative_and_unlabel': 0, 'only_negative': 0},
  'only_unlabel': 0,
  'ambiguous': {'n_sequences': 0}}}

- Exporting data

In [33]:
os.makedirs(output_folder, exist_ok=True)

In [34]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [35]:
negative.shape

(0, 13)

In [36]:
only_negative.shape

(0, 13)

In [37]:
positive.shape

(5, 13)

In [38]:
only_positive.shape

(5, 13)

In [39]:
only_unlabel.shape

(0, 13)

In [40]:
df_ambiguous.shape

(0, 14)

In [41]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)